In [ ]:
import math as maths

import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

import easygems.healpix as egh

In [ ]:
output_vn = 'v6.7.cf_compliance'
deploy = 'dev'
sim = 'glm.n2560_RAL3p3.tuned'
zoom = 10  # Only zoom with any data in so far.
on_jasmin = True

In [ ]:
def load_data(output_vn, deploy, sim, freq, on_jasmin):
    if on_jasmin:
        protocol = 'http'
        baseurl = 'hackathon-o.s3.jc.rl.ac.uk'
    else:
        protocol = 'https'
        baseurl = 'hackathon-o.s3-ext.jc.rl.ac.uk'
    url = f'{protocol}://{baseurl}/sim-data/{deploy}/{output_vn}/{sim}/um.{freq}.hp_z{zoom}.zarr'
    print(url)
    
    ds = xr.open_dataset(url, engine='zarr')
    return ds.pipe(egh.attach_coords)

In [ ]:
ds1h = load_data(output_vn, deploy, sim, 'PT1H', on_jasmin)
ds3h = load_data(output_vn, deploy, sim, 'PT3H', on_jasmin)

In [ ]:
ds1h

In [ ]:
egh.healpix_show(ds1h.sftlf)

In [ ]:
data = [
    [name, da.attrs['standard_name'], da.attrs['units'], da.shape] 
    for name, da in ds1h.data_vars.items()
]

df = pd.DataFrame(data, columns=['vid', 'standard_name', 'units', 'shape'])
df

In [ ]:
ds1plot = ds1h.isel(time=1).compute()

In [ ]:
def plot_all_fields(ds_plot):
    """Plot all fields for a given dataset. Assumes that each field is 2D - i.e. sel(time=..., [pressure=...]) has been applied"""
    zoom = ds_plot.crs.attrs['refinement_level']
    projection = ccrs.Robinson(central_longitude=0)
    # Do not plot orog, land surf.
    plot_vars = [(name, da) for name, da in ds_plot.data_vars.items() if name not in {'orog', 'sftlf'}]
    rows = maths.ceil(len(plot_vars) / 5)
    fig, axes = plt.subplots(rows, 5, figsize=(30, rows * 20 / 6), subplot_kw={'projection': projection}, layout='constrained')
    if 'pressure' in ds_plot.coords:
        plt.suptitle(f'{ds_plot.simulation} z{zoom} @{float(ds_plot.pressure)}hPa')
    else:
        plt.suptitle(f'{ds_plot.simulation} z{zoom}')
            
    for ax, (name, da) in zip(axes.flatten(), plot_vars):
        print(name)
        if name == 'mrsol':
            da = da.isel(depth=0)
            name = 'mrsol@depth=0'
        time = pd.Timestamp(ds_plot.time.values.item())
    
        if abs(da.max() + da.min()) / (da.max() - da.min()) < 0.5:
            # data looks like it needs a diverging cmap.
            # figure out some nice bounds.
            pl, pu = np.percentile(da.values[~np.isnan(da.values)], [2, 98])
            vmax = np.abs([pl, pu]).max()
            kwargs = dict(
                cmap='bwr',
                vmin=-vmax,
                vmax=vmax,
            )
        else:
            kwargs = {}
        ax.set_title(f'time: {time} - {name}')
        ax.set_global()
        im = egh.healpix_show(da, ax=ax, **kwargs);
        long_name = da.long_name
            
        plt.colorbar(im, label=f'{long_name} ({da.attrs.get("units", "-")})')
        ax.coastlines()

In [ ]:
plot_all_fields(ds1plot)

In [ ]:
ds3plot = ds3h.isel(time=1, pressure=20).compute()

In [ ]:
plot_all_fields(ds3plot)

In [ ]:
ds3h.isel(time=1).clw.mean(dim='healpix_index').plot(y='pressure', yincrease=False)

In [ ]:
ds1h

In [ ]:
ds3h